# Requirement 20: Restricted outbound access

> Prevent arbitrary internet access, package installation, and metadata-endpoint access.

**Covered by:** Sandbox

**How it is fulfilled:** The sandbox egress policy is deny-by-default. Only hosts on an explicit allow-list are reachable. Decisions are logged.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()
print("default action:", sandbox.get().egress_policy.default_action)
def code(url):
    return sandbox.exec(f"curl -s -o /dev/null -m 10 -w '%{{http_code}}' {url}").stdout.strip()
blocked = code("https://example.com")
print("example.com ->", blocked)
check("arbitrary internet is blocked", blocked != "200")

default action: Deny
example.com -> 403
PASS  arbitrary internet is blocked


In [3]:
pip = sandbox.exec("pip install requests-toolbelt 2>&1 | tail -2")
print(pip.stdout)
check("package installation from the internet is blocked", "Successfully installed" not in pip.stdout)
meta = sandbox.exec("curl -s -m 5 -H 'Metadata: true' http://169.254.169.254/metadata/instance?api-version=2021-02-01; echo exit=$?")
print("metadata endpoint ->", meta.stdout.strip()[:120])
check("cloud metadata endpoint is unreachable", "compute" not in meta.stdout)

ERROR: Could not find a version that satisfies the requirement requests-toolbelt (from versions: none)
ERROR: No matching distribution found for requests-toolbelt

PASS  package installation from the internet is blocked
metadata endpoint -> exit=7
PASS  cloud metadata endpoint is unreachable


In [4]:
sandbox.add_egress_host_rule("example.com", action="Allow")
import time; time.sleep(3)
allowed = code("https://example.com")
print("example.com after allow rule ->", allowed)
check("allow-listed host is reachable", allowed == "200")
check("other hosts stay blocked", code("https://www.bing.com") != "200")
cleanup(sandbox)

example.com after allow rule -> 200
PASS  allow-listed host is reachable
PASS  other hosts stay blocked


**Caveat:** Egress decisions appear in the audit with about 30 seconds of delay and include host, method and path, which is why the orchestrator's log policy must treat them as sensitive (requirement 23).